#  Mount Drive & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Thesis_V4

!pip install -q -r requirements.txt

Mounted at /content/drive
/content/drive/MyDrive/Thesis_V4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 128.5 MB/s eta 0:00:00


# Imports

In [ ]:
import os
import shutil
import pandas as pd
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix,
)
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

import transformers
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device       :', device)
print('PyTorch      :', torch.__version__)
print('Transformers :', transformers.__version__)

PLOT_DIR = '/content/drive/MyDrive/Thesis_V4/plots'
os.makedirs(PLOT_DIR, exist_ok=True)

def savefig(name):
    p = f'{PLOT_DIR}/{name}.pdf'
    plt.savefig(p, format='pdf', bbox_inches='tight', transparent=True)
    print(f'  Saved → {p}')

Device       : cuda
PyTorch      : 2.10.0+cu128
Transformers : 5.5.4


In [ ]:
class FixedTrainer(Trainer):

    def training_step(
        self,
        model: torch.nn.Module,
        inputs: dict,
        num_items_in_batch=None,
    ) -> torch.Tensor:

        model.train()
        inputs = self._prepare_inputs(inputs)

        with self.compute_loss_context_manager():
            loss = self.compute_loss(model, inputs)

        if self.args.n_gpu > 1:
            loss = loss.mean()

        if getattr(self, 'do_grad_scaling', False):
            self.scaler.scale(loss).backward()
        elif getattr(self, 'use_apex', False):
            from apex import amp as apex_amp
            with apex_amp.scale_loss(loss, self.optimizer) as scaled:
                scaled.backward()
        else:
            self.accelerator.backward(loss)

        return loss.detach() / self.args.gradient_accumulation_steps

print('FixedTrainer defined.')

FixedTrainer defined.


# Load Data

In [ ]:
df = pd.read_csv('all_veltri.csv', index_col=0)
df['AMP']     = df['AMP'].astype(bool).astype(int)
df            = df.sample(frac=1, random_state=42).reset_index(drop=True)
df['seq_len'] = df['aa_seq'].str.len()

print('Shape         :', df.shape)
print('Label counts  :')
print(df['AMP'].value_counts().rename({0: 'Non-AMP (0)', 1: 'AMP (1)'}).to_string())
print(f'Positive rate : {df["AMP"].mean()*100:.1f} %')
print(f'Seq-len range : {df["seq_len"].min()} – {df["seq_len"].max()} aa')

Shape         : (3556, 4)
Label counts  :
AMP
AMP (1)        1778
Non-AMP (0)    1778
Positive rate : 50.0 %
Seq-len range : 11 – 183 aa


# EDA Plot 1 - Overall Label Distribution

In [ ]:
counts = df['AMP'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    ['Non-AMP (0)', 'AMP (1)'], counts.values,
    color=['#4878D0', '#EE854A'], edgecolor='black', linewidth=0.7
)
for bar, v in zip(bars, counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2, v + 18,
        f'{v}\n({v / len(df) * 100:.1f} %)',
        ha='center', fontsize=10, fontweight='bold'
    )
ax.set_ylabel('Number of Sequences', fontsize=13, fontweight='bold')
ax.set_title('Overall Label Distribution', fontsize=15, fontweight='bold')
ax.tick_params(labelsize=11)
ax.set_ylim(0, max(counts.values) * 1.22)
ax.grid(axis='y', alpha=0.35)
plt.tight_layout()
savefig('01_label_distribution')
plt.show()

  Saved → /content/drive/MyDrive/Thesis_V4/plots/01_label_distribution.pdf


# EDA Plot 2 - Sequence Length Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for label, color, name in [(0, '#4878D0', 'Non-AMP'), (1, '#EE854A', 'AMP')]:
    axes[0].hist(
        df[df['AMP'] == label]['seq_len'],
        bins=50, alpha=0.6, color=color, label=name, edgecolor='none'
    )
axes[0].axvline(200, color='red', linestyle='--', lw=1.8, label='Truncation (200)')
axes[0].set_xlabel('Sequence Length', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=13, fontweight='bold')
axes[0].set_title('Sequence Length by Class', fontsize=15, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].tick_params(labelsize=11)
axes[0].grid(True, alpha=0.3)

bp = axes[1].boxplot(
    [df[df['AMP'] == 0]['seq_len'].values, df[df['AMP'] == 1]['seq_len'].values],
    patch_artist=True, medianprops=dict(color='black', linewidth=2)
)
for patch, color in zip(bp['boxes'], ['#4878D0', '#EE854A']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_xticklabels(['Non-AMP (0)', 'AMP (1)'])
axes[1].set_ylabel('Sequence Length', fontsize=13, fontweight='bold')
axes[1].set_title('Length Spread by Class', fontsize=15, fontweight='bold')
axes[1].tick_params(labelsize=11)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
savefig('02_sequence_length')
plt.show()

  Saved → /content/drive/MyDrive/Thesis_V4/plots/02_sequence_length.pdf


# EDA Plot 3 - Amino Acid Composition

In [ ]:
def aa_freq(seqs):
    c = Counter(''.join(seqs))
    t = sum(c.values())
    return {aa: c[aa] / t * 100 for aa in sorted(c)}

freq_amp    = aa_freq(df[df['AMP'] == 1]['aa_seq'])
freq_nonamp = aa_freq(df[df['AMP'] == 0]['aa_seq'])
all_aas     = sorted(set(freq_amp) | set(freq_nonamp))
x, w        = np.arange(len(all_aas)), 0.38

fig, ax = plt.subplots(figsize=(16, 4))
ax.bar(x - w/2, [freq_nonamp.get(a, 0) for a in all_aas], w,
       label='Non-AMP', color='#4878D0', alpha=0.85)
ax.bar(x + w/2, [freq_amp.get(a, 0) for a in all_aas], w,
       label='AMP',     color='#EE854A', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(all_aas, fontsize=11, fontweight='bold')
ax.set_xlabel('Amino Acid', fontsize=13, fontweight='bold')
ax.set_ylabel('Frequency (%)', fontsize=13, fontweight='bold')
ax.set_title('Amino Acid Composition — AMP vs Non-AMP', fontsize=15, fontweight='bold')
ax.legend(fontsize=11)
ax.tick_params(axis='y', labelsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
savefig('03_amino_acid_frequency')
plt.show()

  Saved → /content/drive/MyDrive/Thesis_V4/plots/03_amino_acid_frequency.pdf


# 60 : 20 : 20 Stratified Split

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.4, stratify=df['AMP'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['AMP'], random_state=42
)
print('Train     :', len(train_df))
print('Validation:', len(val_df))
print('Test      :', len(test_df))

Train     : 2133
Validation: 711
Test      : 712


# EDA Plot 4 - Class Balance Across Splits

In [ ]:
split_names = ['Train', 'Validation', 'Test']
splits      = [train_df, val_df, test_df]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

x, w = np.arange(3), 0.32
for i, (label, color, name) in enumerate([(0,'#4878D0','Non-AMP'),(1,'#EE854A','AMP')]):
    vals = [s['AMP'].value_counts().get(label, 0) for s in splits]
    axes[0].bar(x + (i - 0.5)*w, vals, w, label=name, color=color, alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(split_names, fontsize=11, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=13, fontweight='bold')
axes[0].set_title('Class Counts per Split', fontsize=15, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].tick_params(axis='y', labelsize=11)
axes[0].grid(axis='y', alpha=0.3)

pos_pct = [s['AMP'].mean()*100 for s in splits]
bars = axes[1].bar(split_names, pos_pct,
                   color=['#6AAFE6','#AACCDD','#88BBCC'],
                   edgecolor='black', linewidth=0.7)
for bar, v in zip(bars, pos_pct):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+0.5,
                 f'{v:.1f} %', ha='center', fontsize=11, fontweight='bold')
axes[1].axhline(df['AMP'].mean()*100, color='red', linestyle='--',
                lw=1.4, label='Overall positive rate')
axes[1].set_ylim(0, max(pos_pct)*1.4)
axes[1].set_ylabel('AMP Positive Rate (%)', fontsize=13, fontweight='bold')
axes[1].set_title('Positive Rate per Split', fontsize=15, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].tick_params(labelsize=11)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
savefig('04_class_balance_splits')
plt.show()

  Saved → /content/drive/MyDrive/Thesis_V4/plots/04_class_balance_splits.pdf


# Dataset

In [ ]:
print('Loading tokeniser…')
tokenizer = BertTokenizer.from_pretrained('Rostlab/prot_bert_bfd', do_lower_case=False, use_fast=False)
print('Tokeniser ready.')


class AMPData(Dataset):
    def __init__(self, df, tokenizer, max_len=200):
        self.seqs    = list(df['aa_seq'])
        self.labels  = list(df['AMP'].astype(int))
        self.tok     = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        seq = ' '.join(self.seqs[idx])   # ProtBERT needs space-separated AAs
        enc = self.tok(
            seq, truncation=True, padding='max_length', max_length=self.max_len
        )
        item = {k: torch.tensor(v) for k, v in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item


train_dataset = AMPData(train_df, tokenizer)
val_dataset   = AMPData(val_df,   tokenizer)
test_dataset  = AMPData(test_df,  tokenizer)
print(f'train:{len(train_dataset)}  val:{len(val_dataset)}  test:{len(test_dataset)}')

Loading tokeniser…


tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokeniser ready.
train:2133  val:711  test:712


# Metrics

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    probs  = torch.softmax(
        torch.tensor(pred.predictions, dtype=torch.float32), dim=-1
    )[:, 1].numpy()

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )
    acc = accuracy_score(labels, preds)
    try:
        auc = roc_auc_score(labels, probs)
    except Exception:
        auc = float('nan')
    return {
        'accuracy' : acc,
        'f1'       : f1,
        'precision': precision,
        'recall'   : recall,
        'roc_auc'  : auc,
    }

# Model Init

In [ ]:
def model_init():
    return BertForSequenceClassification.from_pretrained(
        'Rostlab/prot_bert_bfd',
        num_labels=2,
        ignore_mismatched_sizes=True,
    )

In [ ]:
# Hyperparameter

In [ ]:
# learning_rates  = [1e-5, 2e-5, 5e-5]
learning_rates = [1e-6, 5e-6, 1e-5, 2e-5, 5e-5, 1e-4]
max_epochs      = 20

best_f1         = 0.0
best_lr         = None
best_model_path = '/content/drive/MyDrive/Thesis_V4/best_model'
os.makedirs(best_model_path, exist_ok=True)

all_histories   = {}   # run-label → per-epoch metric arrays
all_val_metrics = []   # one row per LR

# Training

In [ ]:
import math
_steps_per_epoch = math.ceil(len(train_dataset) / 32)
_total_steps     = max_epochs * _steps_per_epoch
warmup_steps     = max(1, int(0.06 * _total_steps))
print(f"Warmup steps: {warmup_steps} / {_total_steps} total steps")

for lr in learning_rates:
    run_label  = f'lr={lr:.0e}'
    output_dir = f'/content/tmp_ckpt_{run_label}'
    os.makedirs(output_dir, exist_ok=True)

    print(f'\n=== Training  {run_label} ===')

    training_args = TrainingArguments(
        output_dir                  = output_dir,
        num_train_epochs            = max_epochs,
        learning_rate               = lr,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 32,
        weight_decay                = 0.01,
        warmup_steps                = warmup_steps,
        logging_steps               = 50,

        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        save_total_limit            = 1,

        load_best_model_at_end      = True,
        metric_for_best_model       = 'f1',
        greater_is_better           = True,

        fp16                        = torch.cuda.is_available(),
        seed                        = 42,
        report_to                   = 'none',
    )

    trainer = FixedTrainer(
        model_init      = model_init,
        args            = training_args,
        train_dataset   = train_dataset,
        eval_dataset    = val_dataset,
        compute_metrics = compute_metrics,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
    )

    try:
        trainer.train()

        tl, vm = {}, {}
        for e in trainer.state.log_history:
            ep = e.get('epoch')
            if ep is None:
                continue
            ep = round(ep)
            if 'loss' in e and 'eval_loss' not in e:
                tl[ep] = e['loss']
            if 'eval_loss' in e:
                vm[ep] = e

        eps = sorted(set(tl) | set(vm))
        g   = lambda ep, k: vm.get(ep, {}).get(k, float('nan'))

        all_histories[run_label] = {
            'epochs'    : eps,
            'train_loss': [tl.get(e, float('nan')) for e in eps],
            'val_loss'  : [g(e, 'eval_loss')        for e in eps],
            'val_f1'    : [g(e, 'eval_f1')          for e in eps],
            'val_acc'   : [g(e, 'eval_accuracy')    for e in eps],
            'val_prec'  : [g(e, 'eval_precision')   for e in eps],
            'val_rec'   : [g(e, 'eval_recall')      for e in eps],
            'val_auc'   : [g(e, 'eval_roc_auc')     for e in eps],
        }


        val_metrics = {}
        for entry in reversed(trainer.state.log_history):
            if 'eval_loss' in entry:
                val_metrics = dict(entry)
                break
        val_metrics['lr'] = lr
        all_val_metrics.append(val_metrics)
        print('Validation metrics:', val_metrics)

        if val_metrics['eval_f1'] > best_f1:
            best_f1 = val_metrics['eval_f1']
            best_lr = lr
            trainer.save_model(best_model_path)
            tokenizer.save_pretrained(best_model_path)
            best_trainer = trainer
            print(f'  ★  New best model saved to {best_model_path}  (F1={best_f1:.4f})')

    finally:

        if os.path.exists(output_dir):
            shutil.rmtree(output_dir)
            print(f'  Temp checkpoints removed: {output_dir}')

print(f'\nBest Validation F1 : {best_f1:.4f}   (lr={best_lr})')
print(f'Only one model saved at: {best_model_path}')

Warmup steps: 80 / 1340 total steps

=== Training  lr=1e-06 ===


config.json:   0%|          | 0.00/361 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/487 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Validation metrics: {'eval_loss': 0.5913811326026917, 'eval_accuracy': 0.7313642756680732, 'eval_f1': 0.7728894173602854, 'eval_precision': 0.668724279835391, 'eval_recall': 0.9154929577464789, 'eval_roc_auc': 0.7472582687134041, 'eval_runtime': 9.6436, 'eval_samples_per_second': 73.727, 'eval_steps_per_second': 9.229, 'epoch': 3.0, 'step': 201, 'lr': 0.0001}
  Temp checkpoints removed: /content/tmp_ckpt_lr=1e-04

Best Validation F1 : 0.9165   (lr=1e-06)
Only one model saved at: /content/drive/MyDrive/Thesis_V4/best_model


# Training Plot 5 - Learning Curves

In [ ]:
for run_label, h in all_histories.items():
    eps = h['epochs']
    if not eps:
        continue

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle(f'Learning Curves  —  {run_label}', fontsize=16, fontweight='bold')

    axes[0,0].plot(eps, h['train_loss'], 'b-o', ms=4, lw=1.8, label='Train Loss')
    axes[0,0].plot(eps, h['val_loss'],   'r-o', ms=4, lw=1.8, label='Val Loss')
    axes[0,0].fill_between(eps, h['train_loss'], h['val_loss'], alpha=0.08, color='purple')
    axes[0,0].set_title('Loss (train vs validation)', fontsize=13, fontweight='bold')
    axes[0,0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
    axes[0,0].set_ylabel('Cross-Entropy Loss', fontsize=12, fontweight='bold')
    axes[0,0].legend(fontsize=10); axes[0,0].tick_params(labelsize=10)
    axes[0,0].xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    axes[0,0].grid(True, alpha=0.3)

    axes[0,1].plot(eps, h['val_f1'], 'g-o', ms=4, lw=1.8, label='Val F1')
    axes[0,1].set_title('Validation F1 Score', fontsize=13, fontweight='bold')
    axes[0,1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
    axes[0,1].set_ylabel('F1', fontsize=12, fontweight='bold')
    axes[0,1].set_ylim(0, 1); axes[0,1].legend(fontsize=10)
    axes[0,1].tick_params(labelsize=10)
    axes[0,1].xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    axes[0,1].grid(True, alpha=0.3)

    axes[1,0].plot(eps, h['val_prec'], 'm-o', ms=4, lw=1.8, label='Precision')
    axes[1,0].plot(eps, h['val_rec'],  'c-o', ms=4, lw=1.8, label='Recall')
    axes[1,0].set_title('Validation Precision & Recall', fontsize=13, fontweight='bold')
    axes[1,0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
    axes[1,0].set_ylabel('Score', fontsize=12, fontweight='bold')
    axes[1,0].set_ylim(0, 1); axes[1,0].legend(fontsize=10)
    axes[1,0].tick_params(labelsize=10)
    axes[1,0].xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    axes[1,0].grid(True, alpha=0.3)

    axes[1,1].plot(eps, h['val_acc'], 'k-o',             ms=4, lw=1.8, label='Accuracy')
    axes[1,1].plot(eps, h['val_auc'], color='darkorange', ms=4, lw=1.8,
                   marker='o', label='ROC-AUC')
    axes[1,1].set_title('Validation Accuracy & ROC-AUC', fontsize=13, fontweight='bold')
    axes[1,1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
    axes[1,1].set_ylabel('Score', fontsize=12, fontweight='bold')
    axes[1,1].set_ylim(0, 1); axes[1,1].legend(fontsize=10)
    axes[1,1].tick_params(labelsize=10)
    axes[1,1].xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    axes[1,1].grid(True, alpha=0.3)

    plt.tight_layout()
    safe = run_label.replace('=','').replace(' ','_')
    savefig(f'05_learning_curves_{safe}')
    plt.show()

  Saved → /content/drive/MyDrive/Thesis_V4/plots/05_learning_curves_lr1e-06.pdf
  Saved → /content/drive/MyDrive/Thesis_V4/plots/05_learning_curves_lr5e-06.pdf
  Saved → /content/drive/MyDrive/Thesis_V4/plots/05_learning_curves_lr1e-05.pdf
  Saved → /content/drive/MyDrive/Thesis_V4/plots/05_learning_curves_lr2e-05.pdf
  Saved → /content/drive/MyDrive/Thesis_V4/plots/05_learning_curves_lr5e-05.pdf
  Saved → /content/drive/MyDrive/Thesis_V4/plots/05_learning_curves_lr1e-04.pdf


# Training Plot 6 - All LR Runs Overlaid

In [ ]:
run_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('All LR Runs — Validation Metrics vs Epoch', fontsize=15, fontweight='bold')

for ax, (key, title, ylabel) in zip(axes, [
    ('val_loss', 'Validation Loss',    'Loss'),
    ('val_f1',   'Validation F1',      'F1'),
    ('val_auc',  'Validation ROC-AUC', 'AUC'),
]):
    for color, (rl, h) in zip(run_colors, all_histories.items()):
        if h['epochs']:
            ax.plot(h['epochs'], h[key], '-o', color=color, ms=4, lw=1.8, label=rl)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=12, fontweight='bold')
    ax.legend(fontsize=10); ax.tick_params(labelsize=10)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.grid(True, alpha=0.3)

plt.tight_layout()
savefig('06_all_runs_overlaid')
plt.show()

  Saved → /content/drive/MyDrive/Thesis_V4/plots/06_all_runs_overlaid.pdf


# Training Plot 7 - Hyperparameter Comparison Bar Chart

In [ ]:
metric_keys   = ['eval_f1','eval_accuracy','eval_precision','eval_recall','eval_roc_auc']
metric_labels = ['F1','Accuracy','Precision','Recall','ROC-AUC']
run_labels_bar = [f"lr={r['lr']:.0e}" for r in all_val_metrics]

n_runs = len(all_val_metrics)
w = min(0.8 / n_runs, 0.22)
offsets = np.linspace(-(n_runs-1)/2, (n_runs-1)/2, n_runs) * w
x = np.arange(len(metric_labels))
fig, ax = plt.subplots(figsize=(12, 5))
for i, (row, rl, col) in enumerate(zip(all_val_metrics, run_labels_bar, run_colors)):
    vals = [row.get(k, float('nan')) for k in metric_keys]
    bars = ax.bar(x + offsets[i], vals, w, label=rl, color=col, alpha=0.85)
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(bar.get_x()+bar.get_width()/2, v+0.005,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold', rotation=45)
ax.set_xticks(x); ax.set_xticklabels(metric_labels, fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.15); ax.set_ylabel('Score', fontsize=13, fontweight='bold')
ax.set_title('Final Validation Metrics — LR Comparison', fontsize=15, fontweight='bold')
ax.legend(fontsize=11); ax.tick_params(axis='y', labelsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
savefig('07_hyperparameter_comparison')
plt.show()

  Saved → /content/drive/MyDrive/Thesis_V4/plots/07_hyperparameter_comparison.pdf


# Load Best Model

In [ ]:
best_model = BertForSequenceClassification.from_pretrained(
    best_model_path, ignore_mismatched_sizes=True
)
eval_trainer = FixedTrainer(
    model           = best_model,
    compute_metrics = compute_metrics,
)
print(f'Best model loaded  |  lr={best_lr}  |  Val F1={best_f1:.4f}')

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

Best model loaded  |  lr=1e-06  |  Val F1=0.9165


# Test Evaluation

In [ ]:
test_out    = eval_trainer.predict(test_dataset)
test_labels = test_out.label_ids
test_preds  = test_out.predictions.argmax(-1)
test_probs  = torch.softmax(
    torch.tensor(test_out.predictions, dtype=torch.float32), dim=-1
)[:, 1].numpy()

test_metrics = test_out.metrics
print('\nTest metrics:', test_metrics)


Test metrics: {'test_loss': 0.3765779137611389, 'test_model_preparation_time': 0.0295, 'test_accuracy': 0.9030898876404494, 'test_f1': 0.9032258064516129, 'test_precision': 0.9019607843137255, 'test_recall': 0.9044943820224719, 'test_roc_auc': 0.9593722383537433, 'test_runtime': 46.4191, 'test_samples_per_second': 15.339, 'test_steps_per_second': 1.917}


# Plot 8 - Threshold Optimisation on Validation Set

In [ ]:
val_out    = eval_trainer.predict(val_dataset)
val_labels = val_out.label_ids
val_probs  = torch.softmax(
    torch.tensor(val_out.predictions, dtype=torch.float32), dim=-1
)[:, 1].numpy()

thresholds = np.linspace(0.05, 0.95, 181)
f1_vals, prec_vals, rec_vals, acc_vals = [], [], [], []

for t in thresholds:
    p_t = (val_probs >= t).astype(int)
    p, r, f, _ = precision_recall_fscore_support(
        val_labels, p_t, average='binary', zero_division=0
    )
    f1_vals.append(f);   prec_vals.append(p)
    rec_vals.append(r);  acc_vals.append(accuracy_score(val_labels, p_t))

best_t_idx     = int(np.argmax(f1_vals))
OPTIMAL_THRESH = float(thresholds[best_t_idx])

print(f'Optimal threshold : {OPTIMAL_THRESH:.3f}')
print(f'  Val F1          : {f1_vals[best_t_idx]:.4f}')
print(f'  Val Precision   : {prec_vals[best_t_idx]:.4f}')
print(f'  Val Recall      : {rec_vals[best_t_idx]:.4f}')
print(f'  Val Accuracy    : {acc_vals[best_t_idx]:.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, f1_vals,   color='#2ca02c', lw=2.4, label='F1 Score')
ax.plot(thresholds, prec_vals, color='#1f77b4', lw=1.8, linestyle='--', label='Precision')
ax.plot(thresholds, rec_vals,  color='#d62728', lw=1.8, linestyle='--', label='Recall')
ax.plot(thresholds, acc_vals,  color='#9467bd', lw=1.6, linestyle=':',  label='Accuracy')
ax.axvline(OPTIMAL_THRESH, color='black', linestyle='-.', lw=2.0,
           label=f'Optimal  t = {OPTIMAL_THRESH:.3f}')
ax.scatter([OPTIMAL_THRESH], [f1_vals[best_t_idx]], color='black', zorder=6, s=100,
           label=f'Best F1 = {f1_vals[best_t_idx]:.4f}')
ax.axvline(0.5, color='grey', linestyle=':', lw=1.4, label='Default  t = 0.500')
ax.set_xlabel('Classification Threshold', fontsize=13, fontweight='bold')
ax.set_ylabel('Score', fontsize=13, fontweight='bold')
ax.set_title('Threshold Optimisation on Validation Set', fontsize=15, fontweight='bold')
ax.set_xlim(0.05, 0.95); ax.set_ylim(0, 1.06)
ax.legend(loc='best', fontsize=11); ax.tick_params(labelsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
savefig('08_threshold_optimisation_val')
plt.show()

Optimal threshold : 0.480
  Val F1          : 0.9207
  Val Precision   : 0.9259
  Val Recall      : 0.9155
  Val Accuracy    : 0.9212
  Saved → /content/drive/MyDrive/Thesis_V4/plots/08_threshold_optimisation_val.pdf


# Training Plot 9 - Best Run Val Metrics vs Epoch

In [ ]:
best_label = f'lr={best_lr:.0e}'
h   = all_histories[best_label]
eps = h['epochs']

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(eps, h['val_acc'],  '-o', color='#1f77b4',   ms=5, lw=2,   label='Accuracy')
ax.plot(eps, h['val_f1'],   '-s', color='#2ca02c',   ms=5, lw=2,   label='F1 Score')
ax.plot(eps, h['val_prec'], '-^', color='#9467bd',   ms=5, lw=1.7, label='Precision')
ax.plot(eps, h['val_rec'],  '-D', color='#d62728',   ms=5, lw=1.7, label='Recall')
ax.plot(eps, h['val_auc'],  '-P', color='darkorange', ms=5, lw=1.7, label='ROC-AUC')
ax.set_xlabel('Epoch', fontsize=13, fontweight='bold')
ax.set_ylabel('Score', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.set_title(f'Validation Metrics per Epoch — Best Run  ({best_label})', fontsize=15, fontweight='bold')
ax.legend(fontsize=11); ax.tick_params(labelsize=11)
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.grid(True, alpha=0.3)
plt.tight_layout()
savefig('09_best_run_val_metrics')
plt.show()

# Training Plot 10 - Train vs Validation Loss (Best Run)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(eps, h['train_loss'], 'b-o', ms=5, lw=2, label='Training Loss')
ax.plot(eps, h['val_loss'],   'r-o', ms=5, lw=2, label='Validation Loss')
ax.fill_between(eps, h['train_loss'], h['val_loss'], alpha=0.10, color='purple', label='Gap')
ax.set_xlabel('Epoch', fontsize=13, fontweight='bold')
ax.set_ylabel('Cross-Entropy Loss', fontsize=13, fontweight='bold')
ax.set_title(f'Train vs Validation Loss — Best Run  ({best_label})', fontsize=15, fontweight='bold')
ax.legend(fontsize=11); ax.tick_params(labelsize=11)
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.grid(True, alpha=0.3)
plt.tight_layout()
savefig('10_train_val_loss')
plt.show()

  Saved → /content/drive/MyDrive/Thesis_V4/plots/10_train_val_loss.pdf


# Test Plot 11 - Confusion Matrix  (threshold = 0.5)

In [ ]:
cm     = confusion_matrix(test_labels, test_preds)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
annot  = np.array([[f'{v}\n({cm_pct[i,j]:.1f} %)' for j,v in enumerate(row)]
                    for i, row in enumerate(cm)])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax,
            xticklabels=['Non-AMP (0)','AMP (1)'],
            yticklabels=['Non-AMP (0)','AMP (1)'], linewidths=0.5)
ax.set_xlabel('Predicted Label', fontsize=13, fontweight='bold')
ax.set_ylabel('True Label',      fontsize=13, fontweight='bold')
ax.set_title('Confusion Matrix — Test Set  (threshold = 0.500)', fontsize=14, fontweight='bold')
ax.tick_params(labelsize=11)
plt.tight_layout()
savefig('11_confusion_matrix_default')
plt.show()

  Saved → /content/drive/MyDrive/Thesis_V4/plots/11_confusion_matrix_default.pdf


# Test Plot 12 - Confusion Matrix  (optimal threshold)

In [ ]:
test_preds_opt = (test_probs >= OPTIMAL_THRESH).astype(int)

cm2     = confusion_matrix(test_labels, test_preds_opt)
cm2_pct = cm2.astype(float) / cm2.sum(axis=1, keepdims=True) * 100
annot2  = np.array([[f'{v}\n({cm2_pct[i,j]:.1f} %)' for j,v in enumerate(row)]
                     for i, row in enumerate(cm2)])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm2, annot=annot2, fmt='', cmap='Oranges', ax=ax,
            xticklabels=['Non-AMP (0)','AMP (1)'],
            yticklabels=['Non-AMP (0)','AMP (1)'], linewidths=0.5)
ax.set_xlabel('Predicted Label', fontsize=13, fontweight='bold')
ax.set_ylabel('True Label',      fontsize=13, fontweight='bold')
ax.set_title(
    f'Confusion Matrix — Test Set  (threshold = {OPTIMAL_THRESH:.3f}, optimised)',
    fontsize=14, fontweight='bold')
ax.tick_params(labelsize=11)
plt.tight_layout()
savefig('12_confusion_matrix_optimal')
plt.show()

  Saved → /content/drive/MyDrive/Thesis_V4/plots/12_confusion_matrix_optimal.pdf


# Test Plot 13 - ROC Curve

In [ ]:
fpr, tpr, roc_thresh = roc_curve(test_labels, test_probs)
roc_auc = roc_auc_score(test_labels, test_probs)
j_idx   = int(np.argmax(tpr - fpr))

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC  (AUC = {roc_auc:.4f})')
ax.plot([0,1],[0,1], 'navy', lw=1, linestyle='--', label='Random classifier')
ax.scatter(fpr[j_idx], tpr[j_idx], color='red', zorder=6, s=80,
           label=(
               f"Youden's J  t={roc_thresh[j_idx]:.3f}\n"
               f"TPR={tpr[j_idx]:.3f}  FPR={fpr[j_idx]:.3f}"
           ))
ax.fill_between(fpr, tpr, alpha=0.08, color='darkorange')
ax.set_xlim([0,1]); ax.set_ylim([0,1.05])
ax.set_xlabel('False Positive Rate  (1 − Specificity)', fontsize=13, fontweight='bold')
ax.set_ylabel('True Positive Rate  (Sensitivity)',      fontsize=13, fontweight='bold')
ax.set_title('ROC Curve — Test Set', fontsize=15, fontweight='bold')
ax.legend(loc='lower right', fontsize=11); ax.tick_params(labelsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
savefig('13_roc_curve_test')
plt.show()
print(f'ROC-AUC = {roc_auc:.4f}')

  Saved → /content/drive/MyDrive/Thesis_V4/plots/13_roc_curve_test.pdf
ROC-AUC = 0.9594


# Test Plot 14 - Precision Recall Curve

In [ ]:
prec_c, rec_c, pr_thresh = precision_recall_curve(test_labels, test_probs)
avg_prec = average_precision_score(test_labels, test_probs)
no_skill = test_labels.sum() / len(test_labels)
f1_pr    = 2*prec_c[:-1]*rec_c[:-1] / (prec_c[:-1]+rec_c[:-1]+1e-9)
pr_best  = int(np.argmax(f1_pr))

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(rec_c, prec_c, color='blue', lw=2, label=f'PR curve  (AP={avg_prec:.4f})')
ax.axhline(no_skill, color='red', linestyle='--',
           label=f'No-skill baseline  ({no_skill:.2f})')
ax.scatter(rec_c[pr_best], prec_c[pr_best], color='green', zorder=6, s=80,
           label=f'Best-F1 point  (t={pr_thresh[pr_best]:.3f})')
ax.fill_between(rec_c, prec_c, alpha=0.08, color='blue')
ax.set_xlim([0,1]); ax.set_ylim([0,1.05])
ax.set_xlabel('Recall', fontsize=13, fontweight='bold')
ax.set_ylabel('Precision', fontsize=13, fontweight='bold')
ax.set_title('Precision-Recall Curve — Test Set', fontsize=15, fontweight='bold')
ax.legend(loc='best', fontsize=11); ax.tick_params(labelsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
savefig('14_precision_recall_curve_test')
plt.show()
print(f'Average Precision = {avg_prec:.4f}')

# Test Plot 15 - Predicted Probability Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(test_probs[test_labels==0], bins=40, alpha=0.65,
        color='#4878D0', label='Non-AMP  (True=0)', edgecolor='none')
ax.hist(test_probs[test_labels==1], bins=40, alpha=0.65,
        color='#EE854A', label='AMP  (True=1)',     edgecolor='none')
ax.axvline(0.5, color='black', linestyle='--', lw=1.8, label='Default  t=0.500')
ax.axvline(OPTIMAL_THRESH, color='red', linestyle='-.', lw=2.0,
           label=f'Optimal  t={OPTIMAL_THRESH:.3f}')
ax.set_xlabel('Predicted Probability — AMP Class', fontsize=13, fontweight='bold')
ax.set_ylabel('Count', fontsize=13, fontweight='bold')
ax.set_title('Predicted Probability Distribution — Test Set', fontsize=15, fontweight='bold')
ax.legend(fontsize=11); ax.tick_params(labelsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
savefig('15_probability_distribution_test')
plt.show()

# Test Plot 16 - Default vs Optimal Threshold

In [ ]:
p_opt, r_opt, f_opt, _ = precision_recall_fscore_support(
    test_labels, test_preds_opt, average='binary', zero_division=0
)
acc_opt = accuracy_score(test_labels, test_preds_opt)

metric_names = ['Accuracy','F1','Precision','Recall','ROC-AUC']
vals_default = [
    test_metrics.get('test_accuracy',  float('nan')),
    test_metrics.get('test_f1',        float('nan')),
    test_metrics.get('test_precision', float('nan')),
    test_metrics.get('test_recall',    float('nan')),
    roc_auc,
]
vals_optimal = [acc_opt, f_opt, p_opt, r_opt, roc_auc]

x, w = np.arange(len(metric_names)), 0.35
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x-w/2, vals_default, w, label='Threshold = 0.500  (default)',
            color='#5B9BD5', alpha=0.85)
b2 = ax.bar(x+w/2, vals_optimal, w,
            label=f'Threshold = {OPTIMAL_THRESH:.3f}  (optimised)',
            color='#EE854A', alpha=0.85)
for bars in [b1, b2]:
    for bar in bars:
        v = bar.get_height()
        if not np.isnan(v):
            ax.text(bar.get_x()+bar.get_width()/2, v+0.006,
                    f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(metric_names, fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.15); ax.set_ylabel('Score', fontsize=13, fontweight='bold')
ax.set_title('Test Set — Default vs Optimised Threshold', fontsize=15, fontweight='bold')
ax.legend(fontsize=11); ax.tick_params(axis='y', labelsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
savefig('16_default_vs_optimal_threshold')
plt.show()

#  Plot 17 - Validation vs Test Metrics (Best Model)

In [ ]:
best_val_row = next(r for r in all_val_metrics if r['lr'] == best_lr)
val_vals = [best_val_row.get(f'eval_{k}', float('nan'))
            for k in ['accuracy','f1','precision','recall','roc_auc']]

x, w = np.arange(len(metric_names)), 0.35
fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x-w/2, val_vals,     w, label='Validation', color='#5B9BD5', alpha=0.85)
b2 = ax.bar(x+w/2, vals_default, w, label='Test',       color='#ED7D31', alpha=0.85)
for bars in [b1, b2]:
    for bar in bars:
        v = bar.get_height()
        if not np.isnan(v):
            ax.text(bar.get_x()+bar.get_width()/2, v+0.005,
                    f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(metric_names, fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.12); ax.set_ylabel('Score', fontsize=13, fontweight='bold')
ax.set_title(f'Validation vs Test — Best Model  ({best_label})', fontsize=15, fontweight='bold')
ax.legend(fontsize=11); ax.tick_params(axis='y', labelsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
savefig('17_val_vs_test')
plt.show()

# Final Summary

In [ ]:
print('  FINAL RESULTS')
print('=' * 65)
print(f'Best LR            : {best_lr}')
print(f'Best Val F1        : {best_f1:.4f}')
print(f'Optimal threshold  : {OPTIMAL_THRESH:.3f}  (tuned on validation set)')
print(f'Model saved to     : {best_model_path}')
print()
print(f'{"Metric":<18} {"Default (0.500)":>18} {f"Optimal ({OPTIMAL_THRESH:.3f})": >18}')
print('-' * 56)
for name, vd, vo in zip(metric_names, vals_default, vals_optimal):
    print(f'{name:<18} {vd:>18.4f} {vo:>18.4f}')
print()
print('Plots saved to:', PLOT_DIR)
for f in sorted(os.listdir(PLOT_DIR)):
    if f.endswith('.pdf'):
        print(f'  {f}')